In [20]:
from langchain_google_genai import ChatGoogleGenerativeAI 
from dotenv import load_dotenv
from pydantic import BaseModel , Field 
from typing import Literal 
from langchain_core.output_parsers import PydanticOutputParser , StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableBranch , RunnableLambda

In [4]:
load_dotenv()

model = ChatGoogleGenerativeAI(model = 'gemini-3.1-flash-lite-preview')

In [6]:
class Person(BaseModel):

    sentiment : Literal['Negative' , 'Positive'] = Field(description='Check the Sentiment of the FeedBack')

    Gender : Literal['Male' , 'Female'] =  Field(description = 'Check whether the Gender is given or not ',  default='Female')

In [11]:
pydantic_parser  = PydanticOutputParser(pydantic_object=Person)

string_parser = StrOutputParser()

In [13]:
prompt_1 = PromptTemplate(
    template='Extract the Gender and check the sentiment of the following review \n {review} \n {format_instruction}' , 
    input_variables=['review'] , 
    partial_variables={'format_instruction':pydantic_parser.get_format_instructions()}
)

In [ ]:
classifier_chain = prompt_1 | model | pydantic_parser

In [28]:
prompt_M_P =  PromptTemplate(template='Generate the best product as a suggestion for a Male and also thanks for the following review and also just Product name \n {review}')

prompt_M_N =  PromptTemplate(template='Generate the best product as a suggestion and also sorry for the incovinience for the product , just Product name as per the review given below \n {review}')

prompt_F_P =  PromptTemplate(template='Generate the best product as a suggestion for Female and also thanks for the following review and also just Product name \n {review}')

prompt_F_N =  PromptTemplate(template='Generate the best product as a suggestion for female  and also sorry for the incovinience for the product , just Product name as per the review given below \n {review}')

In [29]:
branch_chain = RunnableBranch(

    (lambda x: x.Gender == "Male" and x.sentiment == "Positive", prompt_M_P | model | string_parser ),

    (lambda x: x.Gender == "Male" and x.sentiment == "Negative",  prompt_M_N | model | string_parser),

    (lambda x: x.Gender == "Female" and x.sentiment == "Positive", prompt_F_P | model | string_parser),

    (lambda x: x.Gender == "Female" and x.sentiment == "Negative",prompt_F_N | model | string_parser ),

    RunnableLambda(lambda x : ' Not able to find the snetiment and gender' )
)

In [30]:
chain = classifier_chain | branch_chain

In [39]:
text = 'This is a worst face Product i got lot\'s acne now it is a bad mamaearth facewash'

In [40]:
classifier_response = classifier_chain.invoke({'review' : text})

response = chain.invoke({'review' : text})

In [41]:
print(classifier_response.sentiment)
print(classifier_response.Gender)
print(response)

Negative
Female
Since you did not provide the specific review text, I have selected a highly-rated, versatile product that consistently performs well for female consumers across various categories.

**Suggested Product:**
**Dyson Airwrap Multi-Styler**

*(Note: We sincerely apologize for the inconvenience caused by the previous product experience. We strive for excellence and hope this suggestion better meets your needs.)*
